# ASL Landmark MLP Training
Train an MLP on MediaPipe hand landmarks (63 features) after removing the `nothing` class.

In [3]:
import sys

print(sys.version)
print(sys.executable)

3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
C:\Users\bchar\AppData\Local\Programs\Python\Python312\python.exe


In [4]:
import sys

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install tensorflow


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 7.1 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.1.1
    Uninstalling pip-26.1.1:
      Successfully uninstalled pip-26.1.1
   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
   ---------------------------------------- 1.0/350.9 MB 8.4 MB/s eta 0:00:42
   ---------------------------------------- 2.9/350.9 MB 7.3 MB/s eta 0:00:48
    --------------------------------------- 4.5/350.9 MB 7.3 MB/s eta 0:00:48
    --------------------------------------- 5.8/350.9 MB 7.3 MB/s eta 0:00:48
    --------------------------------------- 7.3/350.9 MB 7.3 MB/s eta 0:00:48
   - -------------------------------------- 8.9/350.9 MB 7.3 MB/s eta 0:00:47
   - -------------------------------------- 10.5/350.9 MB 7.3 MB/s eta 0:00:47
   - ----------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.72.0 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
google-api-core 2.28.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.19.5, but you have protobuf 7.35.1 which is incompatible.
google-cloud-aiplatform 1.126.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
google-cloud-appengine-logging 1.7.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but y

In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Load CSV
df = pd.read_csv("landmarks.csv")

print("Original shape:", df.shape)
print(df['label'].value_counts())

In [ ]:
# Remove 'nothing'
df = df[df["label"] != "nothing"].reset_index(drop=True)

print("After removing 'nothing':", df.shape)
print(df["label"].value_counts())

In [ ]:
X = df.drop(columns=["label"]).values.astype("float32")
y = df["label"].values

encoder = LabelEncoder()
y = encoder.fit_transform(y)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Classes:", encoder.classes_)
print("Number of classes:", len(encoder.classes_))
print("Feature shape:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
model = Sequential([
    Dense(256, activation="relu", input_shape=(63,)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation="relu"),

    Dense(28, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        verbose=1
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=128,
    callbacks=callbacks
)

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Accuracy: {acc*100:.2f}%")
print(f"Test Loss: {loss:.4f}")

In [ ]:
model.save("asl_mlp_model.keras")
print("Saved model: asl_mlp_model.keras")
print("Saved encoder: label_encoder.pkl")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()